# ขอบเขตเพิ่มเติม: เปรียบเทียบ PM2.5 ที่วัดเองกับสถานี PCD ที่ใกล้ที่สุด

**วัตถุประสงค์:** ตรวจสอบว่าค่า PM2.5 ที่เก็บด้วยเครื่องมือภาคสนาม (32 จุด) มีความสอดคล้อง/เป็นไปในทิศทางเดียวกันหรือไม่ เมื่อเทียบกับค่าที่วัดได้จากสถานีอ้างอิงของกรมควบคุมมลพิษ (PCD) ที่อยู่ใกล้ที่สุด (สถานี 24T ต.หน้าพระลาน และ 25T ต.ปากเพรียว จ.สระบุรี) เพื่อใช้เป็นหลักฐานสนับสนุนความน่าเชื่อถือของข้อมูล (data reliability / concurrent validity)

**ข้อควรระวัง (สำคัญ):**
- การเปรียบเทียบนี้เป็นการตรวจสอบ "ทิศทาง/ความสอดคล้องเชิงสัมพัทธ์" ไม่ใช่การสอบเทียบ (calibration) เนื่องจากเครื่องมือภาคสนามเก็บค่าแบบ spot reading 1 นาที ขณะที่สถานี PCD รายงานค่าเฉลี่ยรายชั่วโมง/รายวัน และตำแหน่งของสถานี PCD กับจุดเก็บข้อมูลภาคสนามก็ไม่ได้อยู่ที่เดียวกัน (มีระยะห่างจริง)
- Air4Thai API ที่ใช้ดึงข้อมูลย้อนหลัง (getHistoryData.php) เป็น API ที่ไม่มีเอกสารทางการ อาจมีการเปลี่ยนแปลงได้ ถ้าเรียกไม่สำเร็จ notebook นี้มี fallback ให้ upload ไฟล์ CSV ที่ export ด้วยมือจากเว็บ https://air4thai.pcd.go.th/webV3/#/History แทน

**ขั้นตอน:** ติดตั้งไลบรารี → อัปโหลดข้อมูลภาคสนามดิบ → ค้นหาพิกัดสถานี PCD และคำนวณระยะทาง → ดึง/อัปโหลดข้อมูลย้อนหลังของสถานี PCD → จับคู่ข้อมูล → วิเคราะห์ความสัมพันธ์ → ดาวน์โหลดผลลัพธ์

## 1. ติดตั้งไลบรารีและ import

In [ ]:
# SECTION 0: Setup
!pip install -q requests pandas numpy scipy matplotlib openpyxl

import re, time, json
import numpy as np
import pandas as pd
import requests
import openpyxl
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files

plt.rcParams['figure.dpi'] = 120
print('พร้อมใช้งาน ✅')

## 2. อัปโหลดไฟล์ข้อมูลภาคสนามต้นฉบับ

อัปโหลดไฟล์ `บันทึกการตรวจวัดปริมาณฝุ่น.xlsx` (ไฟล์เดียวกับที่ใช้ใน Table1/Table2/Figure1 notebooks)

In [ ]:
import re, time, json
import numpy as np
import pandas as pd
import requests
import openpyxl
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files

uploaded = files.upload()
RAW_XLSX = list(uploaded.keys())[0]
print('อัปโหลดไฟล์แล้ว:', RAW_XLSX)

## 3. ตรวจสอบโครงสร้างคอลัมน์ก่อนดึงข้อมูล

⚠️ **สำคัญ**: เซลล์นี้แสดงข้อมูลดิบ 5 แถวแรกแบบไม่ตัดคอลัมน์ เพื่อให้คุณยืนยัน index ของคอลัมน์ วันที่ และ **เวลา** (ถ้ามี) ก่อนไปขั้นตอนถัดไป — notebook ก่อนหน้านี้ (Table1/Table2/Figure1) ใช้ index คงที่ (name=0, url=1, address=2, pm25=6, tpm=9, date=13) แต่ไม่ได้ดึงคอลัมน์เวลา ถ้าไฟล์มีคอลัมน์เวลาแยกต่างหาก ให้สังเกต index แล้วแก้ `COL_TIME` ในเซลล์ถัดไป

In [ ]:
wb = openpyxl.load_workbook(RAW_XLSX, data_only=True)
ws = wb['Sheet1']

print('Header row (row 1-2):')
for r in ws.iter_rows(min_row=1, max_row=2, values_only=True):
    print(r)
print()
print('First 3 data rows (from row 3):')
for r in ws.iter_rows(min_row=3, max_row=5, values_only=True):
    print(r)

## 4. ตั้งค่า column index และแยกข้อมูลดิบ (parse)

👉 แก้ `COL_TIME` ด้านล่างให้ตรงกับผลลัพธ์ที่เห็นใน Section 3 ถ้าไฟล์มีคอลัมน์เวลาการวัด ใส่ index ที่ถูกต้อง หรือใส่ `None` ถ้าไม่มี (จะเปรียบเทียบแบบค่าเฉลี่ยรายวันแทน)

In [ ]:
# 👉 ปรับ index คอลัมน์ตรงนี้หากจำเป็น (ยึดตาม convention จาก notebook ก่อนหน้า)
COL_NAME = 0
COL_URL = 1
COL_ADDRESS = 2
COL_PM25 = 6
COL_DATE = 13
COL_TIME = None  # <-- แก้เป็น index ที่ถูกต้อง (int) ถ้าพบคอลัมน์เวลาใน Section 3

records = {}
order = []
for row in ws.iter_rows(min_row=3, max_row=1001, values_only=True):
    name = row[COL_NAME]
    if not name:
        continue
    if name not in records:
        records[name] = {
            'name': name,
            'url': row[COL_URL],
            'address': row[COL_ADDRESS],
            'pm25': row[COL_PM25],
            'date': str(row[COL_DATE]),
            'time': str(row[COL_TIME]) if COL_TIME is not None else None,
        }
        order.append(name)

sites_raw = [records[n] for n in order]
print(f'จำนวนจุดตรวจวัดทั้งหมด: {len(sites_raw)}')
assert len(sites_raw) == 32, 'คาดว่าจะมี 32 จุด - ตรวจสอบไฟล์ต้นฉบับถ้าไม่ตรง'

## 5. Resolve พิกัด GPS ของจุดภาคสนาม (จาก Google Maps link)

In [ ]:
PATTERNS = [
    re.compile(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)"),
    re.compile(r"@(-?\d+\.\d+),(-?\d+\.\d+)"),
    re.compile(r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)"),
    re.compile(r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"),
]

def extract_latlon(url_or_text):
    for pat in PATTERNS:
        m = pat.search(url_or_text)
        if m:
            return float(m.group(1)), float(m.group(2))
    return None, None

def resolve_link(short_url, timeout=15):
    try:
        r = requests.get(short_url, allow_redirects=True, timeout=timeout,
                          headers={'User-Agent': 'Mozilla/5.0'})
        lat, lon = extract_latlon(r.url)
        if lat is None:
            lat, lon = extract_latlon(r.text[:5000])
        return lat, lon
    except Exception:
        return None, None

for i, s in enumerate(sites_raw, 1):
    lat, lon = resolve_link(s['url'])
    s['lat'], s['lon'] = lat, lon
    print(f"[{i:02d}/32] {s['name']}: {'OK' if lat else 'FAILED'}")
    time.sleep(0.4)

df = pd.DataFrame(sites_raw)
missing = df[df['lat'].isna()]
if len(missing):
    print('\n⚠️ จุดที่ resolve พิกัดไม่สำเร็จ ต้องเติมด้วยมือก่อนคำนวณระยะทาง:')
    print(missing[['name']])
df.head()

## 6. ค้นหาพิกัดสถานี PCD (24T, 25T) และคำนวณระยะทางจากแต่ละจุด

In [ ]:
# ดึงพิกัดสถานีปัจจุบันทั้งหมดจาก Air4Thai (มี lat/lon ของทุกสถานี รวมถึง 24t, 25t)
PCD_STATIONS = ['24t', '25t']  # 24t = ต.หน้าพระลาน อ.เฉลิมพระเกียรติ, 25t = ต.ปากเพรียว อ.เมือง (สระบุรี)

station_coords = {}
try:
    resp = requests.get('http://air4thai.pcd.go.th/services/getNewAQI_JSON.php', timeout=20)
    stations_json = resp.json()['stations']
    for st in stations_json:
        sid = st.get('stationID', '').lower()
        if sid in PCD_STATIONS:
            station_coords[sid] = {
                'lat': float(st['lat']), 'lon': float(st['long']),
                'nameTH': st.get('nameTH'), 'nameEN': st.get('nameEN'),
            }
    print('พบพิกัดสถานี:', station_coords)
except Exception as e:
    print('⚠️ ดึงพิกัดสถานีอัตโนมัติไม่สำเร็จ:', e)
    print('กรุณาใส่พิกัดด้วยมือในเซลล์ถัดไป (fallback)')

# --- Fallback พิกัดโดยประมาณ (ใช้เฉพาะถ้าการดึงอัตโนมัติด้านบนล้มเหลว) ---
# ตรวจสอบพิกัดจริงกับเว็บ air4thai.pcd.go.th ก่อนใช้งานจริง เพราะเป็นค่าประมาณการเบื้องต้นเท่านั้น
if '24t' not in station_coords:
    station_coords['24t'] = {'lat': 14.6300, 'lon': 100.9130, 'nameTH': 'ต.หน้าพระลาน (ประมาณ - โปรดตรวจสอบ)'}
if '25t' not in station_coords:
    station_coords['25t'] = {'lat': 14.5300, 'lon': 100.9100, 'nameTH': 'ต.ปากเพรียว (ประมาณ - โปรดตรวจสอบ)'}

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

df = df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
for sid, info in station_coords.items():
    df[f'dist_to_{sid}_km'] = haversine_km(df['lat'], df['lon'], info['lat'], info['lon'])

dist_cols = [f'dist_to_{sid}_km' for sid in station_coords]
df['nearest_pcd_station'] = df[dist_cols].idxmin(axis=1).str.replace('dist_to_', '').str.replace('_km', '')
df['dist_to_nearest_pcd_km'] = df[dist_cols].min(axis=1)

df[['name', 'pm25', 'date', 'nearest_pcd_station', 'dist_to_nearest_pcd_km']]

## 7. ดึงข้อมูลย้อนหลังของสถานี PCD (24T, 25T) ช่วงแคมเปญ

พยายามดึงผ่าน Air4Thai historical API ก่อน (ไม่มีเอกสารทางการ อาจใช้ไม่ได้เสมอไป) — ถ้าล้มเหลว ให้ไป **Section 7B** เพื่อ upload ไฟล์ CSV ที่ export ด้วยมือแทน

In [ ]:
# 👉 ปรับช่วงวันที่ให้ตรงกับแคมเปญของคุณ (ค่าเริ่มต้นอิงจาก Section 2.1 ของต้นฉบับ: 5-29 มิ.ย. 2023)
SDATE = '2023-06-05'
EDATE = '2023-06-29'

def fetch_pcd_history(station_id, sdate=SDATE, edate=EDATE, param='PM25'):
    url = 'http://air4thai.pcd.go.th/services/getHistoryData.php'
    params = {'stationID': station_id, 'param': param, 'type': 'hour',
              'sdate': sdate, 'edate': edate, 'stime': '00', 'etime': '23'}
    r = requests.get(url, params=params, timeout=30)
    data = r.json()
    # โครงสร้างผลลัพธ์ของ API นี้อาจเปลี่ยนแปลงได้ - ปรับ parsing ตรงนี้ถ้าจำเป็น
    rows = data.get('stations', [{}])[0].get(param, [])
    out = pd.DataFrame(rows)
    if not out.empty:
        out['stationID'] = station_id
    return out

pcd_history = {}
api_ok = True
for sid in station_coords:
    try:
        dfh = fetch_pcd_history(sid)
        if dfh.empty:
            raise ValueError('empty result')
        pcd_history[sid] = dfh
        print(f'{sid}: ดึงสำเร็จ {len(dfh)} แถว')
    except Exception as e:
        print(f'{sid}: ❌ ดึงไม่สำเร็จ ({e}) -> ใช้ Section 7B แทน')
        api_ok = False

print('\nสถานะโดยรวม:', 'API ใช้งานได้' if api_ok else 'ต้อง upload ไฟล์ manual (Section 7B)')

### 7B. Fallback: อัปโหลดไฟล์ที่ export จาก Air4Thai (ใช้เฉพาะถ้า Section 7 ล้มเหลว)

จากไฟล์ตัวอย่างที่คุณ export มา (`2023.xlsx`) พบว่า Air4Thai ส่งออกเป็น **ไฟล์เดียวรวมทุกสถานีทั้งเครือข่าย แบบค่าเฉลี่ยรายวัน** ไม่ใช่รายชั่วโมง โดยมีโครงสร้างดังนี้:
- ชีตชื่อ `PM2.5`
- คอลัมน์แรกชื่อ `Date` ตามด้วยคอลัมน์รหัสสถานีทุกสถานี (เช่น `24T`, `25T`, ...) เป็นค่า PM2.5 เฉลี่ยรายวันของแต่ละสถานี
- ค่าที่ไม่มีข้อมูลจะเป็นข้อความ `N/A`
- ท้ายไฟล์มีแถวหมายเหตุ (ไม่ใช่วันที่) ซึ่งเซลล์ด้านล่างจะกรองออกให้อัตโนมัติ

**อัปโหลดไฟล์ .xlsx นี้ได้เลย (ไฟล์เดียว ไม่ต้องแยกไฟล์ต่อสถานี)** — ถ้าไฟล์ของคุณมีชื่อชีตหรือโครงสร้างต่างออกไป ให้ปรับ `SHEET_NAME`/ชื่อคอลัมน์ในเซลล์ถัดไป

⚠️ ข้อมูลชุดนี้เป็น**ค่าเฉลี่ยรายวัน** ไม่มีความละเอียดรายชั่วโมง ดังนั้นขั้นตอนถัดไปจะเปรียบเทียบแบบ **ค่าเฉลี่ยรายวัน** เท่านั้น (การเทียบแบบชั่วโมงเดียวกันใน Section 9 จะถูกข้ามอัตโนมัติ)

In [ ]:
from google.colab import files

SHEET_NAME = 'PM2.5'

daily_means = {}  # จะถูกเติมโดยตรงจากไฟล์ manual (ไม่ผ่าน pcd_history/hourly แบบ Section 7)
manual_mode = False

if not api_ok:
    print('อัปโหลดไฟล์ Excel ที่ export จาก Air4Thai (ไฟล์เดียว รวมทุกสถานี เช่น 2023.xlsx):')
    manual_uploaded = files.upload()
    fname = list(manual_uploaded.keys())[0]

    raw_pcd = pd.read_excel(fname, sheet_name=SHEET_NAME)
    # กรองแถวที่ไม่ใช่วันที่ออก (เช่นแถวหมายเหตุท้ายไฟล์)
    raw_pcd = raw_pcd[pd.to_datetime(raw_pcd['Date'], errors='coerce').notna()].copy()
    raw_pcd['Date'] = pd.to_datetime(raw_pcd['Date'])

    mask = (raw_pcd['Date'] >= pd.to_datetime(SDATE)) & (raw_pcd['Date'] <= pd.to_datetime(EDATE))
    campaign_pcd = raw_pcd.loc[mask].copy()
    print(f'พบข้อมูล {len(campaign_pcd)} วัน ในช่วง {SDATE} ถึง {EDATE}')

    for sid in station_coords:
        col = sid.upper()  # ไฟล์ export ใช้ตัวพิมพ์ใหญ่ เช่น '24T', '25T'
        if col not in campaign_pcd.columns:
            print(f'⚠️ ไม่พบคอลัมน์ {col} ในไฟล์ - ตรวจสอบชื่อคอลัมน์ใน raw_pcd.columns')
            continue
        vals = pd.to_numeric(campaign_pcd[col], errors='coerce')  # 'N/A' (ข้อความ) -> NaN
        daily_means[sid] = pd.Series(vals.values, index=campaign_pcd['Date'].dt.date)
        n_ok = daily_means[sid].notna().sum()
        print(f'{sid} ({col}): มีค่า {n_ok}/{len(campaign_pcd)} วันในช่วงแคมเปญ')

    manual_mode = True
    print('\n✅ ใช้ข้อมูลรายวันจากไฟล์ manual แล้ว (ข้าม hourly matching ใน Section 9 โดยอัตโนมัติ)')
else:
    print('⏭️ ข้ามขั้นตอนนี้ - API สำเร็จแล้วใน Section 7')

## 8. จับคู่ข้อมูลภาคสนามกับข้อมูลสถานี PCD ที่ใกล้ที่สุด

ℹ️ ถ้าใช้ไฟล์ manual จาก Section 7B (`manual_mode = True`) ขั้นตอนนี้ใช้ `daily_means` ที่สร้างไว้แล้วโดยตรง ไม่ต้องแก้อะไรเพิ่ม

⚠️ ถ้า Section 7 (API) สำเร็จ ให้ตรวจสอบโครงสร้างคอลัมน์ของ `pcd_history[sid]` ก่อนรันเซลล์คำนวณ แล้วแก้ `PCD_DATETIME_COL` และ `PCD_VALUE_COL` ให้ตรงกับผลลัพธ์จริง

In [ ]:
if manual_mode:
    print('โหมด manual (จากไฟล์รายวัน) - ตัวอย่าง daily_means:')
    for sid, s in daily_means.items():
        print(f'--- {sid} ---')
        print(s.head(3))
        print()
else:
    for sid, dfh in pcd_history.items():
        print(f'--- {sid} ---')
        print(dfh.head(3))
        print(list(dfh.columns))
        print()

In [ ]:
# 👉 ใช้เฉพาะกรณี API สำเร็จ (Section 7) - ถ้าใช้ไฟล์ manual (Section 7B) ข้ามส่วนนี้ไปเลย เพราะ daily_means ถูกสร้างไว้แล้ว
PCD_DATETIME_COL = 'DATETIMEDATA'  # ปกติ Air4Thai history API ใช้ชื่อนี้ (ตรวจสอบให้แน่ใจกับผลลัพธ์ Section 8 ด้านบน)
PCD_VALUE_COL = 'PM25'

if not manual_mode:
    daily_means = {}
    for sid, dfh in pcd_history.items():
        dfh = dfh.copy()
        dfh[PCD_DATETIME_COL] = pd.to_datetime(dfh[PCD_DATETIME_COL], errors='coerce')
        dfh[PCD_VALUE_COL] = pd.to_numeric(dfh[PCD_VALUE_COL], errors='coerce')
        dfh['date_only'] = dfh[PCD_DATETIME_COL].dt.date
        daily_means[sid] = dfh.groupby('date_only')[PCD_VALUE_COL].mean()
        pcd_history[sid] = dfh
else:
    print('ℹ️ ใช้ daily_means ที่สร้างจากไฟล์ manual ใน Section 7B แล้ว (ไม่รันซ้ำ)')

import re

def normalize_date(date_str):
    """แปลง string วันที่เป็น datetime.date - รองรับรูปแบบ D/M/YY แบบ พ.ศ. 2 หลัก (เช่น '5/6/66' = 5 มิ.ย. 2566 = 2023-06-05)
    และ fallback ไปใช้ pandas (dayfirst=True) พร้อมแปลง พ.ศ.->ค.ศ. อัตโนมัติสำหรับรูปแบบอื่น"""
    s = str(date_str).strip()
    m = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{2,4})$', s)
    if m:
        day, month, yy = int(m.group(1)), int(m.group(2)), int(m.group(3))
        # บางแถวในไฟล์ต้นฉบับพบว่าใส่ปีไม่สม่ำเสมอ (ส่วนใหญ่เป็น พ.ศ. 2 หลัก เช่น 66,
        # แต่บางแถวดันเป็น ค.ศ. 2 หลัก เช่น 23) - ลองทั้งสองแบบแล้วเลือกปีที่สมเหตุสมผล (2015-2035)
        if yy >= 100:
            candidates = [yy]
        else:
            candidates = [yy + 2500 - 543, yy + 2000]  # ลอง พ.ศ.->ค.ศ. ก่อน แล้วลอง ค.ศ. ตรงๆ
        for year in candidates:
            if 2015 <= year <= 2035:
                try:
                    return pd.Timestamp(year=year, month=month, day=day).date()
                except ValueError:
                    continue
        return None
    d = pd.to_datetime(s, dayfirst=True, errors='coerce')
    if pd.isna(d):
        return None
    d = d.date()
    if d.year > 2400:
        d = d.replace(year=d.year - 543)
    return d

def lookup_pcd_daily(sid, date_str):
    d = normalize_date(date_str)
    if d is None:
        return np.nan
    return daily_means.get(sid, pd.Series(dtype=float)).get(d, np.nan)

df['pcd_daily_mean_pm25'] = df.apply(lambda r: lookup_pcd_daily(r['nearest_pcd_station'], r['date']), axis=1)

# ถ้ามีคอลัมน์เวลาในข้อมูลภาคสนาม *และ* มาจาก hourly API เท่านั้น ถึงจะลองจับคู่แบบรายชั่วโมงได้
def lookup_pcd_hourly(sid, date_str, time_str):
    if manual_mode or time_str is None or time_str in ('None', 'nan'):
        return np.nan
    try:
        dt = pd.to_datetime(f'{date_str} {time_str}', errors='coerce')
        dt_hour = dt.floor('h')
        dfh = pcd_history[sid]
        match = dfh.loc[dfh[PCD_DATETIME_COL] == dt_hour, PCD_VALUE_COL]
        return match.iloc[0] if len(match) else np.nan
    except Exception:
        return np.nan

if COL_TIME is not None and not manual_mode:
    df['pcd_same_hour_pm25'] = df.apply(
        lambda r: lookup_pcd_hourly(r['nearest_pcd_station'], r['date'], r['time']), axis=1)
else:
    df['pcd_same_hour_pm25'] = np.nan
    print('ℹ️ ใช้การเปรียบเทียบแบบค่าเฉลี่ยรายวันเท่านั้น (ไม่มีข้อมูลรายชั่วโมงในแหล่งข้อมูลนี้)')

df['diff_daily'] = df['pm25'] - df['pcd_daily_mean_pm25']
df[['name', 'pm25', 'date', 'time', 'nearest_pcd_station', 'dist_to_nearest_pcd_km',
    'pcd_daily_mean_pm25', 'pcd_same_hour_pm25', 'diff_daily']]

In [ ]:
# --- เซลล์ตรวจสอบ (diagnostic) - รันหลัง Section 8 เพื่อดูว่าทำไม match ไม่ติด ---
n_match = df['pcd_daily_mean_pm25'].notna().sum()
print(f'จับคู่ pcd_daily_mean_pm25 สำเร็จ: {n_match} / {len(df)} จุด\n')

print('ตัวอย่างวันที่ในข้อมูลภาคสนาม (df["date"]):')
print(df['date'].head(5).tolist())
print('ชนิดข้อมูล:', type(df['date'].iloc[0]))
print()

print('ค่า nearest_pcd_station ที่พบทั้งหมด:', df['nearest_pcd_station'].unique())
print('keys ที่มีอยู่จริงใน daily_means:', list(daily_means.keys()))
print()

for sid, s in daily_means.items():
    print(f'ตัวอย่าง index (วันที่) ใน daily_means["{sid}"]:', list(s.index[:5]))
    print(f'ชนิดข้อมูลของ index:', type(s.index[0]) if len(s.index) else 'ว่างเปล่า')
    print()

print('ตัวอย่างวันที่หลังแปลงด้วย normalize_date():')
for d in df['date'].head(5):
    print(f'  {d!r}  ->  {normalize_date(d)}')

## 9. วิเคราะห์ความสัมพันธ์และทิศทางความสอดคล้อง

In [ ]:
def report_correlation(x_col, y_col, label):
    sub = df[[x_col, y_col]].dropna()
    n = len(sub)
    if n < 3:
        print(f'{label}: n={n} - ข้อมูลจับคู่ได้ไม่พอสำหรับคำนวณสหสัมพันธ์')
        return
    r, p = stats.pearsonr(sub[x_col], sub[y_col])
    rho, p_rho = stats.spearmanr(sub[x_col], sub[y_col])
    direction = 'สอดคล้องทิศทางเดียวกัน (บวก)' if r > 0 else 'ทิศทางตรงข้าม (ลบ) - ควรตรวจสอบเพิ่มเติม'
    print(f'--- {label} (n={n}) ---')
    print(f'Pearson r  = {r:.3f}, p = {p:.3f}')
    print(f'Spearman rho = {rho:.3f}, p = {p_rho:.3f}')
    print(f'ทิศทาง: {direction}\n')

report_correlation('pm25', 'pcd_daily_mean_pm25', 'ค่าฝุ่นภาคสนาม vs PCD (ค่าเฉลี่ยรายวัน)')
if df['pcd_same_hour_pm25'].notna().sum() >= 3:
    report_correlation('pm25', 'pcd_same_hour_pm25', 'ค่าฝุ่นภาคสนาม vs PCD (ชั่วโมงเดียวกัน)')

In [ ]:
# กราฟ scatter เทียบค่าฝุ่นภาคสนาม vs PCD (ค่าเฉลี่ยรายวัน) พร้อมเส้น 1:1 อ้างอิง
fig, ax = plt.subplots(figsize=(6, 6))
sub = df.dropna(subset=['pm25', 'pcd_daily_mean_pm25'])
colors = sub['nearest_pcd_station'].map({'24t': '#e6550d', '25t': '#3182bd'})
ax.scatter(sub['pcd_daily_mean_pm25'], sub['pm25'], c=colors, edgecolors='black', alpha=0.85)
lims = [0, max(sub['pm25'].max(), sub['pcd_daily_mean_pm25'].max()) * 1.1]
ax.plot(lims, lims, 'k--', alpha=0.5, label='1:1 reference')
ax.set_xlabel('PCD nearest-station daily-mean PM2.5 (µg/m³)')
ax.set_ylabel('Field-measured PM2.5 (µg/m³)')
ax.set_title('Field vs. nearest PCD reference station (daily mean)')
ax.legend()
plt.tight_layout()
plt.savefig('Figure_PCD_crossvalidation.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. ดาวน์โหลดผลลัพธ์

In [ ]:
from google.colab import files

out_cols = ['name', 'pm25', 'date', 'time', 'lat', 'lon', 'nearest_pcd_station',
            'dist_to_nearest_pcd_km', 'pcd_daily_mean_pm25', 'pcd_same_hour_pm25', 'diff_daily']
df[out_cols].to_csv('Table_PCD_crossvalidation.csv', index=False)
df[out_cols].to_excel('Table_PCD_crossvalidation.xlsx', index=False)

files.download('Table_PCD_crossvalidation.csv')
files.download('Table_PCD_crossvalidation.xlsx')
files.download('Figure_PCD_crossvalidation.png')

---
## หมายเหตุสำหรับนำผลไปเขียนในต้นฉบับ

หลังรันเสร็จและได้ค่า Pearson r / Spearman rho / p-value จริงแล้ว บอก Claude ในแชทถัดไปได้เลย จะช่วยร่างข้อความสำหรับ:
- **Methods** หัวข้อใหม่ เช่น `2.11 Cross-validation Against PCD Reference Stations` (อธิบายวิธีจับคู่ข้อมูล ระยะทางเฉลี่ยไปสถานีที่ใกล้ที่สุด)
- **Results** หัวข้อใหม่ เช่น `3.x Cross-validation Against PCD Reference Data` (รายงานค่า r/rho, ทิศทาง, ตารางเปรียบเทียบ, รูป scatter)
- **Limitations** เพิ่มย่อหน้าเรื่องข้อจำกัดของการเปรียบเทียบนี้ (spot reading vs hourly/daily average, ระยะห่างระหว่างจุดวัดกับสถานี PCD, ความแตกต่างของ siting/สภาพแวดล้อมรอบสถานี)

อย่าลืมแนบค่า **ระยะทางเฉลี่ยจากจุดภาคสนามไปสถานี PCD ที่ใกล้ที่สุด** (`dist_to_nearest_pcd_km`) มาด้วย เพราะเป็นตัวเลขสำคัญที่ผู้ทวนสอบ (reviewer) มักถามถึงเมื่อเห็นการเปรียบเทียบลักษณะนี้